# 03 - Fine-tune BanglaBERT

Fine-tune `sagorsarker/bangla-bert-base` for Bangla toxic comment classification.

**Run this notebook on Kaggle or Colab with GPU enabled.**

In [ ]:
# Uncomment on Kaggle/Colab
# !pip install transformers datasets accelerate scikit-learn pyyaml -q

In [ ]:
import torch
import pandas as pd
import json
import os
import sys
sys.path.append('..')

from src.dataset import load_dataset
from src.train import train_with_multiple_runs, train_learning_curve
from src.evaluate import evaluate_model, get_classification_report, get_confusion_matrix, aggregate_runs
from src.model import load_trained_model
from src.utils import load_config, save_metrics, ensure_dirs

config = load_config('../configs/config.yaml')
model_config = config['models']['banglabert']
MODEL_NAME = model_config['name']

ensure_dirs('../results', '../plots', '../saved_models')
print(f'Device: {"cuda" if torch.cuda.is_available() else "cpu"}')
print(f'Model: {MODEL_NAME}')

## Load Dataset

In [ ]:
DATA_PATH = '../data/train.csv'

train_texts, test_texts, train_labels, test_labels = load_dataset(
    DATA_PATH,
    text_column=config['data']['text_column'],
    label_column=config['data']['label_column'],
    test_size=config['data']['test_size'],
    random_seed=config['data']['random_seed'],
)
print(f'Train: {len(train_texts)}, Test: {len(test_texts)}')

## Train (3 Runs with Error Bars)

In [ ]:
all_metrics = train_with_multiple_runs(
    MODEL_NAME,
    train_texts, test_texts,
    train_labels, test_labels,
    model_config,
    output_dir='../results',
    num_runs=config['training']['num_runs'],
)

In [ ]:
summary = aggregate_runs(all_metrics)
print('\n=== BanglaBERT Results (3 runs) ===')
for key, stats in summary.items():
    if 'eval_' in key:
        print(f"  {key}: {stats['mean']:.4f} +/- {stats['std']:.4f}")

save_metrics(
    {'model': 'bangla-bert-base', 'runs': all_metrics, 'summary': summary},
    '../results/banglabert_results.json'
)

## Learning Curve (25%, 50%, 100%)

In [ ]:
lc_results = train_learning_curve(
    MODEL_NAME,
    train_texts, test_texts,
    train_labels, test_labels,
    model_config,
    fractions=[0.25, 0.5, 1.0],
    output_dir='../results',
)

print('\n=== Learning Curve ===')
for r in lc_results:
    print(f"  {r['fraction']*100:.0f}% ({r['num_samples']} samples): "
          f"Acc={r['eval_accuracy']:.4f}, F1={r['eval_f1']:.4f}")

save_metrics(
    {'model': 'bangla-bert-base', 'learning_curve': lc_results},
    '../results/banglabert_learning_curve.json'
)

## Evaluate Best Model

In [ ]:
best_model_dir = '../results/bangla-bert-base/run_1/best_model'
model, tokenizer = load_trained_model(best_model_dir)
if torch.cuda.is_available():
    model = model.cuda()

metrics, preds = evaluate_model(model, tokenizer, test_texts, test_labels)
print('\n=== Best Model Evaluation ===')
for k, v in metrics.items():
    print(f'  {k}: {v:.4f}')

print('\n=== Classification Report ===')
print(get_classification_report(test_labels, preds))

print('\n=== Confusion Matrix ===')
print(get_confusion_matrix(test_labels, preds))